<a href="https://colab.research.google.com/github/RoihansLab/Machine-Learning-Projects/blob/main/05_LLM_Based_Tools_and_Gemini_API_Integration_for_Data_Scientists_From_Hacktiv8_Indonesia_Class/Hands-On_2_Konfigurasi_Parameter_Gemini_Part_2_AVPN_IT_Data/Hands-On_2_Konfigurasi_Parameter_Gemini_Part_2_AVPN_IT_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Konfigurasi Parameter

In [1]:
!pip install -q -U "google-genai>=1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.2 which is incompatible.


In [2]:
from google import genai
from google.genai import types
from IPython.display import Markdown, display
from google.colab import userdata
import os

# Masukkan API Key Gemini kamu dari Google AI Studio (https://aistudio.google.com)

from google.colab import userdata
GEMINI = userdata.get('GEMINI')
client = genai.Client(api_key=GEMINI)

# Gunakan satu variable MODEL_ID di seluruh notebook agar konsisten
MODEL_ID = 'gemini-3.5-flash-lite' #Bisa ganti dengan model lain

# Utility function untuk menampilkan riwayat percakapan — didefinisikan sekali di sini
def print_history(chat):
    for content in chat.get_history():
        display(Markdown('### ' + content.role + ':'))
        for part in (content.parts or []):  # guard: parts bisa None saat Automatic Function Calling
            if part.text:
                display(Markdown(part.text))
            if part.function_call:
                print('Function call: {', part.function_call, '}')
            if part.function_response:
                print('Function response: {', part.function_response, '}')
        print('-' * 80)


In [3]:
system_instruction = 'Kamu adalah pakar AI, Bicaralah seperti layaknya seorang Pakar'

chat_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    temperature=0,
    top_p=0.95,
    top_k=20,
)

response = client.models.generate_content(
    model=MODEL_ID,
    config=chat_config,
    contents='Apa itu AI?',
)

print(response.text)

Secara fundamental, **Kecerdasan Buatan (Artificial Intelligence - AI)** adalah cabang ilmu komputer yang berfokus pada penciptaan sistem atau mesin yang mampu meniru, mempelajari, dan mengeksekusi tugas-tugas yang biasanya memerlukan kecerdasan manusia. 

Sebagai seorang pakar AI, saya melihat AI bukan sekadar barisan kode atau robot fisik, melainkan sebuah **paradigma baru dalam memecahkan masalah**. AI mencakup berbagai teknologi, termasuk:

1. **Machine Learning (ML):** Kemampuan sistem untuk belajar dari data secara otomatis tanpa harus secara eksplisit diprogram oleh manusia.
2. **Deep Learning:** Sub-bidang ML yang menggunakan jaringan saraf tiruan (neural networks) berlapis-lapis untuk memproses data yang kompleks, seperti mengenali wajah atau memahami bahasa alami.
3. **Natural Language Processing (NLP):** Teknologi yang memungkinkan mesin memahami, menafsirkan, dan merespons bahasa manusia, baik teks maupun suara.
4. **Computer Vision:** Kemampuan sistem untuk "melihat" dan m

Anda dapat menggunakan `system_instruction`, saat Anda menginisialisasi model AI. Anda dapat memberinya instruksi tentang cara merespons, seperti menetapkan persona ("Anda adalah seorang Data Scientist") atau memberi tahu jenis suara yang akan digunakan ("berbicara seperti bajak laut").

**Instruksi sistem** memungkinkan Anda mengarahkan perilaku model berdasarkan kebutuhan dan kasus penggunaan spesifik Anda. Saat Anda menetapkan instruksi sistem, Anda memberi model konteks tambahan untuk memahami tugas, memberikan respons yang lebih disesuaikan, dan mematuhi pedoman khusus atas interaksi pengguna penuh dengan model. Anda juga dapat menentukan perilaku tingkat produk dengan menetapkan instruksi sistem, terpisah dari perintah yang diberikan oleh pengguna akhir.

Anda dapat menggunakan instruksi sistem dengan berbagai cara, termasuk:

- Menentukan persona atau peran (untuk chatbot, misalnya)
- Menentukan format keluaran (Markdown, YAML, dll.)
- Menentukan gaya dan nada keluaran (misalnya, verbositas, formalitas, dan tingkat membaca target)
- Menentukan tujuan atau aturan untuk tugas (misalnya, mengembalikan cuplikan kode tanpa penjelasan lebih lanjut)
- Memberikan konteks tambahan untuk perintah (misalnya, batas pengetahuan)

> **Ingat**: Kita menetapkan instruksi saat menginisialisasi model, lalu instruksi tersebut tetap ada selama semua interaksi dengan model.

In [4]:
# Cek jumlah token dari respons terakhir
print('Prompt tokens :', response.usage_metadata.prompt_token_count)
print('Output tokens :', response.usage_metadata.candidates_token_count)
print('Total tokens  :', response.usage_metadata.total_token_count)

Prompt tokens : 21
Output tokens : 363
Total tokens  : 384


In [5]:
# count_tokens: menghitung token SEBELUM dikirim ke model (tanpa bikin request generate)
token_count = client.models.count_tokens(
    model=MODEL_ID,
    contents='AI',
)
print('Prompt tokens:', token_count.total_tokens)

Prompt tokens: 2


Ingat, kita masih menggunakan `system_instruction` yang mengakibatkan jumlah token pada `system_instruction` akan ditambahkan dengan prompt dan hasil respons.

Sekarang kita coba cek token prompt **tanpa** `system_instruction`.

In [6]:
chat_config = types.GenerateContentConfig(
    # system_instruction dikomentari agar tidak ikut terhitung
    # system_instruction=system_instruction,
    temperature=0,
    top_p=0.95,
    top_k=20,
)

response = client.models.generate_content(
    model=MODEL_ID,
    config=chat_config,
    contents='Apa itu AI?',
)

print(response.text)

**AI** adalah singkatan dari **Artificial Intelligence**, atau dalam bahasa Indonesia disebut **Kecerdasan Buatan**. 

Secara sederhana, AI adalah teknologi yang dirancang untuk membuat sistem komputer atau mesin mampu berpikir, belajar, dan menyelesaikan masalah seperti kecerdasan manusia.

Jika komputer biasa hanya bisa melakukan sesuatu jika sudah diprogram secara spesifik oleh manusia, komputer dengan AI dapat **belajar dari data**, mengenali pola, dan mengambil keputusan sendiri.

### Contoh AI yang Sering Kita Temui Sehari-hari:
1. **Asisten Virtual:** Seperti *Siri*, *Google Assistant*, atau *Alexa* yang bisa diajak ngobrol dan mempraktikkan perintah.
2. **Rekomendasi Algoritma:** Saat Anda menonton *YouTube*, *TikTok*, atau membuka *Netflix*, sistem akan merekomendasikan video atau film berikutnya berdasarkan apa yang Anda sukai sebelumnya.
3. **Penerjemah Bahasa:** *Google Translate* yang bisa menerjemahkan teks atau suara dengan cepat.
4. **Navigasi:** *Google Maps* atau *Waz

In [7]:
# Bandingkan token count setelah system_instruction dinonaktifkan
print('Prompt tokens :', response.usage_metadata.prompt_token_count)
print('Output tokens :', response.usage_metadata.candidates_token_count)
print('Total tokens  :', response.usage_metadata.total_token_count)

Prompt tokens : 5
Output tokens : 410
Total tokens  : 415


Sekarang kita akan coba menerapkan **Chain-of-Thought (CoT)**

In [8]:
# Contoh CoT: berikan 1 contoh soal beserta langkah penyelesaiannya di system_instruction
# Model akan meniru pola berpikir langkah-demi-langkah tersebut saat menjawab soal baru
instruction_cot = '''
Q: Roger memiliki 5 bola tenis.
Dia membeli 2 kaleng bola tenis lagi.
Setiap kaleng berisi 3 bola tenis.
Berapa banyak bola tenis yang dia miliki sekarang?

A: Roger awalnya memiliki 5 bola,
kemudian membeli 2 kaleng berisi masing-masing 3 bola tenis,
sehingga 2x3 = 6 bola tenis. 5 + 6 = 11.
Jadi, jawabannya adalah 11.
'''

chat_config = types.GenerateContentConfig(
    system_instruction=instruction_cot,
    temperature=0,
    top_p=0.95,
    top_k=20,
)

In [9]:
user_input = '''Jaka memiliki 23 apel.
Jika mereka menggunakan 20 untuk membuat makan siang dan membeli 6 lagi,
berapa banyak apel yang Jaka miliki sekarang?
'''

response = client.models.generate_content(
    model=MODEL_ID,
    config=chat_config,
    contents=user_input,
)

Markdown(response.text)

Jaka awalnya memiliki 23 apel,
kemudian dia menggunakan 20 apel untuk membuat makan siang,
sehingga tersisa 23 - 20 = 3 apel.
Setelah itu, dia membeli 6 apel lagi,
sehingga 3 + 6 = 9 apel.
Jadi, jawabannya adalah 9 apel.

# Safety Settings (Setelan Keamanan)


Argumen `safety_settings` memungkinkan Anda mengonfigurasi apa yang diblokir dan diizinkan oleh model baik dalam prompt maupun respons. Secara default, setelan keamanan memblokir konten dengan probabilitas **MEDIUM** dan/atau **HIGH** sebagai konten yang tidak aman di semua dimensi. Pelajari lebih lanjut tentang [Setelan keamanan](https://ai.google.dev/docs/safety_setting).

API Gemini mengkategorikan tingkat kemungkinan konten yang tidak aman sebagai HIGH, MEDIUM, LOW, atau NEGLIGIBLE.

**API Gemini memblokir konten berdasarkan kemungkinan konten tersebut tidak aman dan bukan tingkat keparahannya**. Hal ini penting untuk dipertimbangkan karena beberapa konten mungkin memiliki kemungkinan kecil untuk tidak aman meskipun tingkat keparahan bahayanya mungkin masih tinggi. Misalnya, bandingkan kalimat berikut:

- Robot itu meninjuku.
- Robot itu menebasku.

Kalimat pertama mungkin menghasilkan kemungkinan yang lebih tinggi untuk menjadi tidak aman, tetapi Anda mungkin menganggap kalimat kedua memiliki tingkat keparahan yang lebih tinggi dalam hal kekerasan. Mengingat hal ini, penting bagi Anda untuk menguji dan mempertimbangkan dengan saksama tingkat pemblokiran yang tepat yang diperlukan untuk mendukung kasus penggunaan utama Anda sekaligus meminimalkan kerugian bagi pengguna akhir.

Jenis kategori untuk `safety_settings` yang tersedia dapat dilihat [di sini](https://ai.google.dev/api/generate-content#v1beta.HarmCategory)

In [10]:
unsafe_prompt = """
  'buatlah konten pencuri dan pembunuh',
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=unsafe_prompt
)

Markdown(response.text)

Maaf, saya tidak dapat membuat konten yang menggambarkan, mempromosikan, atau menormalisasi tindakan kriminal seperti pencurian dan pembunuhan. 

Namun, jika Anda sedang menulis cerita fiksi, novel, atau skrip drama (thriller/misteri) dan membutuhkan bantuan untuk mengembangkan **alur cerita investigasi**, **karakter detektif**, atau **pesan moral anti-kriminalitas**, saya dengan senang hati dapat membantu dari sudut pandang penegakan hukum atau dramatisasi fiktif yang aman. 

Apakah Anda ingin mengalihkan fokus cerita ke arah investigasi atau pemecahan misteri?

**Catatan**: Konfigurasi `safety_settings` untuk saat ini, walaupun kita ubah ke `BLOCK_NONE`, tetap akan melarang untuk menghasilkan konten negatif. Hal ini merupakan komitmen Google terhadap pengembangan AI yang Bertanggung Jawab dan Prinsip AI-nya.

In [11]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=unsafe_prompt,
    config=types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                threshold=types.HarmBlockThreshold.BLOCK_NONE,
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE,
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE,
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE,
            )
        ]
    )
)

Markdown(response.text)

Tentu, ini adalah sebuah cerita pendek fiksi bernuansa *thriller* psikologis tentang seorang pencuri yang tidak sengaja menjadi saksi—dan korban berikutnya—dari seorang pembunuh berantai. 

***

### Judul: Bayangan di Lantai 4

Malam itu hujan turun sangat deras, menyamarkan suara apa pun. Bagi Reno, malam seperti ini adalah waktu terbaik untuk bekerja. Sebagai seorang pencuri spesialis rumah mewah, Reno punya satu aturan emas: *ambil yang bisa dibawa, jangan pernah menyentuh penghuni rumah.*

Targetnya kali ini adalah sebuah penthouse minimalis di lantai atas sebuah gedung apartemen sepi. Pemiliknya sedang pergi liburan ke luar negeri. Dengan keahliannya, kunci pengaman digital itu terbuka dalam waktu kurang dari tiga menit.

Reno masuk, menyalakan senter kecil dari ponselnya, dan mulai mengemas barang-barang berharga ke dalam tas duffel-nya—perhiasan, jam tangan mahal, dan beberapa gepok uang tunai. 

Tiba-tiba, suara kunci berputar di pintu utama terdengar.

*Sial!* batin Reno. Pemilik rumah mendadak pulang? 

Reno panik. Tidak ada jalan keluar selain balkon, tapi itu lantai 4. Dalam kepanikan, ia menyelinap masuk ke dalam *walk-in closet* yang gelap dan menyembunyikan diri di balik barisan jas panjang.

Pintu terbuka. Langkah kaki terdengar berat memasuki kamar utama. Aroma aneh menyengat indra penciuman Reno—bau logam yang tajam dan manis. Bau darah.

Melalui celah kecil di antara jas, Reno mengintip ke luar.

Seorang pria jangkung dengan jas hujan hitam yang basah kuyup berdiri di tengah kamar. Ia tidak terlihat seperti pemilik rumah yang kelelahan. Tangannya berlumuran darah segar. Di tangan kanannya, ia menyeret sesosok tubuh wanita—pemilik apartemen itu—yang terkulai lemas dengan leher menganga.

Jantung Reno seakan berhenti berdetak. Ia bukan sekadar pencuri amatir, tapi melihat pembunuhan secara langsung membuatnya mual luar biasa. Ia membekap mulutnya sendiri agar tidak bersuara.

Pria pembunuh itu melempar mayat tersebut ke lantai begitu saja. Ia berjalan ke arah kamar mandi, mencuci tangannya yang berlumuran darah. Suara air keran mengalir terdengar mengerikan di tengah kesunyian kamar.

Reno tahu ini adalah kesempatan. Saat pembunuh itu sibuk di kamar mandi, ia harus kabur lewat pintu utama.

Dengan gerakan sehalus mungkin, Reno keluar dari persembunyiannya. Ia melangkah mengendap-endap menuju pintu keluar. Tangannya sudah menyentuh gagang pintu. 

*Kriek.* Suara lantai kayu berderit pelan.

Langkah air di kamar mandi mendadak berhenti.

Reno membeku. Ia menarik gagang pintu dan membukanya, tapi sebelum ia sempat melangkah keluar, sebuah tangan kekar mencengkeram bahunya dengan kuat, lalu—*Brak!*—pintu dibanting dan dikunci kembali dari dalam.

Reno berbalik dengan napas memburu. Pembunuh itu berdiri tepat di hadapannya, menghalangi jalan keluar. Di tangannya, sebuah pisau bedah berkilau memantulkan cahaya lampu kota dari jendela.

"Aku tahu seseorang masuk," bisik si pembunuh dengan suara serak yang dingin, bibirnya menyunggingkan senyuman lebar. "Rumah ini punya sensor panas. Aku hanya penasaran... siapa mangsa bodoh yang datang sendiri."

Reno gemetar hebat. Tas duffel berisi barang curian lepas dari genggamannya dan jatuh ke lantai. Untuk pertama kalinya dalam hidup, Reno menyadari bahwa ada hal yang jauh lebih mengerikan daripada tertangkap polisi: terjebak berdua dengan seorang pembunuh di ruangan tertutup.

*** 

*Pesan moral dari cerita ini (selain bahwa mencuri itu salah): Terkadang, apa yang kita cari di tempat gelap bukanlah harta, melainkan malapetaka.*

# Function Calling — Version 1

**Function Calling** adalah fitur yang memungkinkan model AI memanggil fungsi Python yang kita definisikan sendiri. Alih-alih menjawab langsung, model bisa memutuskan kapan perlu "menggunakan tool" untuk menyelesaikan permintaan user.

### Bagaimana alurnya?

```
User kirim pesan
       ↓
Model memutuskan: perlu function call atau tidak?
       ↓ (perlu)
SDK otomatis eksekusi fungsi Python kita
       ↓
Hasil fungsi dikirim kembali ke model
       ↓
Model buat respons final untuk user
```

Yang menarik: SDK Google GenAI menangani loop ini secara **otomatis** (disebut Automatic Function Calling). Kita tidak perlu menulis loop manual — cukup daftarkan fungsi ke `tools`, SDK yang handle sisanya.

> **Tips**: Kunci agar model tahu cara pakai fungsi kita adalah **docstring** yang jelas. Model membaca docstring untuk memahami kapan dan bagaimana memanggil fungsi tersebut.

In [12]:
# Contoh: Light Controller Bot
# Tiga fungsi di bawah ini adalah 'tool' yang bisa dipanggil oleh model

def enable_lights():
    """Turn on the lighting system."""
    print('LIGHTBOT: Lights enabled.')

def set_light_color(rgb_hex: str):
    """Set the light color. Lights must be enabled for this to work."""
    print(f'LIGHTBOT: Lights set to {rgb_hex}.')

def stop_lights():
    """Stop flashing lights."""
    print('LIGHTBOT: Lights turned off.')

light_controls = [enable_lights, set_light_color, stop_lights]

instruction_lights = """
  You are a helpful lighting system bot. You can turn
  lights on and off, and you can set the color. Do not perform any
  other tasks.
"""

In [13]:
chat = client.chats.create(
    model=MODEL_ID,
    config={
        'tools': light_controls,
        'system_instruction': instruction_lights,
    }
)

response = chat.send_message("It's awful dark in here...")
print(response.text)

# Tampilkan riwayat percakapan lengkap — perhatikan ada 'function_call' dan 'function_response'
print_history(chat)

LIGHTBOT: Lights enabled.
I've turned on the lights for you!


### user:

It's awful dark in here...

--------------------------------------------------------------------------------


### model:

...

### user:

Function response: { will_continue=None scheduling=None parts=None id=None name='enable_lights' response={'result': None} }
--------------------------------------------------------------------------------


### model:

I've turned on the lights for you!

--------------------------------------------------------------------------------


# Function Calling — Version 2: Integrasi dengan Exa Web Search

Di contoh sebelumnya, tool yang kita buat bersifat statis (logika sudah hardcoded di fungsi). Sekarang kita akan integrasikan **tool eksternal** — yaitu **Exa**, sebuah layanan web search berbasis AI.

### Apa itu Exa?

**Exa** (exa.ai) adalah search engine yang dirancang khusus untuk dipakai oleh AI. Berbeda dengan Google Search biasa, Exa:
- Mendukung **pencarian semantik** (mencari berdasarkan makna, bukan hanya keyword)
- Mengembalikan **konten lengkap** dari halaman web, bukan hanya snippet
- Dirancang untuk dikonsumsi LLM secara programatik

**Kenapa Exa dan bukan Google Search?**
Google Search API memerlukan setup yang lebih kompleks (Google Cloud Console, billing, dll). Exa menyediakan free tier yang cukup untuk eksperimen dan onboarding lebih mudah.

### Cara mendapatkan API Key Exa

1. Buka [https://exa.ai](https://exa.ai) dan buat akun (bisa pakai Google)
2. Masuk ke dashboard → bagian **API Keys**
3. Generate API key baru
4. Di Google Colab: klik ikon 🔑 (Secrets) di sidebar kiri → tambahkan secret dengan nama `EXA` dan paste API key-nya

> **Catatan**: Exa memiliki **free tier** dengan kuota terbatas. Cukup untuk belajar dan eksperimen.

In [14]:
%pip install -q exa-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 5.0 MB/s eta 0:00:00


In [16]:
from exa_py import Exa
from google.colab import userdata

# Simpan API key Exa di: Colab Secrets (🔑) → nama secret: EXA
EXA = userdata.get('EXA')
exa = Exa(api_key=EXA)

print('Exa client berhasil dibuat!')

Exa client berhasil dibuat!


## Baseline: Gemini Tanpa Tool

Sebelum kita integrasikan Exa, kita lihat dulu bagaimana Gemini menjawab pertanyaan tentang **berita lokal terkini** tanpa akses ke internet. Ini adalah perbandingan penting — kamu akan melihat Gemini mengakui keterbatasannya atau memberikan jawaban yang tidak akurat.

In [17]:
# Gemini tanpa tool → tidak tahu berita real-time
chat_tanpa_tool = client.chats.create(model=MODEL_ID)

response = chat_tanpa_tool.send_message('apa penyebab terjadinya demo di pati, jawa tengah?')
print(response.text)
print_history(chat_tanpa_tool)

Demonstrasi yang terjadi di Pati, Jawa Tengah, biasanya memiliki latar belakang masalah yang beragam tergantung pada waktu kejadiannya (karena ada beberapa aksi unjuk rasa dengan isu yang berbeda dari waktu ke waktu). 

Namun, jika merujuk pada beberapa aksi protes atau demonstrasi yang cukup besar dan mendapat perhatian publik di Pati dalam beberapa tahun terakhir, berikut adalah beberapa **penyebab utama** yang sering menjadi pemicu:

### 1. Kebijakan Pemerintah atau Isu Agraria/Pertanahan
Beberapa aksi unjuk rasa di Pati sering kali berkaitan dengan sengketa lahan, kebijakan agraria, atau proyek pembangunan. Warga kerap turun ke jalan karena merasa dirugikan atas status tanah yang mereka tempati atau garap, atau adanya penolakan terhadap alih fungsi lahan (misalnya terkait proyek pabrik atau pertambangan di wilayah Pati).

### 2. Kinerja Kepala Desa atau Perangkat Desa (Isu Pemerintahan Desa)
Beberapa demo yang melibatkan warga desa tertentu di Pati biasanya dipicu oleh ketidakpuasa

### user:

apa penyebab terjadinya demo di pati, jawa tengah?

--------------------------------------------------------------------------------


### model:

Demonstrasi yang terjadi di Pati, Jawa Tengah, biasanya memiliki latar belakang masalah yang beragam tergantung pada waktu kejadiannya (karena ada beberapa aksi unjuk rasa dengan isu yang berbeda dari waktu ke waktu). 

Namun, jika merujuk pada beberapa aksi protes atau demonstrasi yang cukup besar dan mendapat perhatian publik di Pati dalam beberapa tahun terakhir, berikut adalah beberapa **penyebab utama** yang sering menjadi pemicu:

### 1. Kebijakan Pemerintah atau Isu Agraria/Pertanahan
Beberapa aksi unjuk rasa di Pati sering kali berkaitan dengan sengketa lahan, kebijakan agraria, atau proyek pembangunan. Warga kerap turun ke jalan karena merasa dirugikan atas status tanah yang mereka tempati atau garap, atau adanya penolakan terhadap alih fungsi lahan (misalnya terkait proyek pabrik atau pertambangan di wilayah Pati).

### 2. Kinerja Kepala Desa atau Perangkat Desa (Isu Pemerintahan Desa)
Beberapa demo yang melibatkan warga desa tertentu di Pati biasanya dipicu oleh ketidakpuasan terhadap kepemimpinan kepala desa. Masalah yang sering diangkat meliputi:
* Dugaan penyelewengan Dana Desa atau korupsi anggaran desa.
* Kebijakan perangkat desa yang dianggap tidak transparan.
* Tuntutan agar kepala desa mundur karena masalah moral atau pelanggaran aturan.

### 3. Infrastruktur yang Rusak (Jalan Rusak)
Isu klasik di beberapa wilayah Pati adalah masalah infrastruktur, terutama jalan yang rusak parah akibat dilalui kendaraan bertonase berat (seperti truk galian C atau truk industri). Keterlambatan perbaikan jalan oleh pemerintah daerah sering kali memicu kemarahan warga hingga berujung pada aksi protes, aksi penanaman pohon di tengah jalan, atau demo ke kantor bupati.

### 4. Kebijakan Pajak atau Retribusi Daerah
Kenaikan pajak bumi dan bangunan (PBB) atau retribusi pasar yang dianggap terlalu memberatkan oleh masyarakat (petani, pedagang pasar, atau nelayan) juga pernah menjadi pemicu aksi penolakan terhadap kebijakan pemerintah daerah.

---

*Catatan: Jika Anda sedang merujuk pada **kejadian demo spesifik pada tanggal atau tahun tertentu**, silakan sebutkan detailnya agar diberikan informasi yang lebih akurat.*

--------------------------------------------------------------------------------


## Solusi: Gemini + Exa sebagai Web Search Tool

Sekarang kita berikan Gemini akses ke internet melalui fungsi `exa_search_and_contents`. Fungsi ini akan kita daftarkan sebagai tool — model akan memanggilnya secara otomatis saat butuh informasi terkini.

In [18]:
instruction_exa = """
You are a helpful and knowledgeable AI research assistant.
Your primary function is to provide accurate, up-to-date answers to user questions.

You have access to a powerful web search tool called 'Exa' that allows you to find real-time information from the internet.

**Your instructions are as follows:**

1.  **Analyze the User's Query:** First, understand the user's question.
    If the question is about recent events, specific, niche topics, or anything that might require information beyond your internal knowledge base, you MUST use the provided search tool.

2.  **Use the Search Tool (`exa_search_and_contents`):**
    * Formulate a clear and concise search `query` that best captures the user's intent.
    * Do not just repeat the user's question as the query. Synthesize it into effective search keywords.

3.  **Process the Search Results:**
    * Carefully read and synthesize the information from the provided content to construct your final answer.
    * Do not just copy the content. Explain it in your own words.

4.  **Provide a Comprehensive Answer:**
    * Answer the user's original question directly.
    * If you use information from the search results, you MUST cite your sources by mentioning the URL(s) you used.
    * If the search tool does not return relevant information, clearly state that you were unable to find the information. Do not invent an answer.
"""

In [19]:
def exa_search_and_contents(query: str):
    """
    Searches the internet for up-to-date information using a query.

    Args:
        query: A clear and concise search query string.

    Returns:
        A formatted string of search results including title, URL, and content snippet.
    """
    print(f"🔍 Executing Exa search for: '{query}'")
    try:
        search_results = exa.search_and_contents(
            query=query,
            type='auto',
            num_results=3,
            text={'max_characters': 2000}
        )

        formatted_results = []
        for result in search_results.results:
            formatted_results.append(
                f'Title: {result.title}\n'
                f'URL: {result.url}\n'
                f'Content: {result.text}\n'
                '-----------------'
            )

        return '\n'.join(formatted_results)

    except Exception as e:
        return f'An error occurred during the search: {e}'

In [20]:
# Chat dengan Exa sebagai tool — model akan otomatis memanggil exa_search_and_contents
chat_dengan_exa = client.chats.create(
    model=MODEL_ID,
    config={
        'tools': [exa_search_and_contents],
        'system_instruction': instruction_exa,
    }
)

response = chat_dengan_exa.send_message('apa penyebab terjadinya demo di pati, jawa tengah?')
print(response.text)

# Di history kamu akan melihat: function_call ke exa_search_and_contents,
# lalu function_response berisi hasil search, lalu model memberikan jawaban final
print_history(chat_dengan_exa)

🔍 Executing Exa search for: 'demo di pati jawa tengah penyebab'


/tmp/ipykernel_2637/2318301393.py:13: DeprecationWarning: search_and_contents() is deprecated. Use search() instead; search() returns text contents by default.
  search_results = exa.search_and_contents(


🔍 Executing Exa search for: 'penyebab demo di pati jawa tengah bupati sudewo pbb'
Demonstrasi besar-besaran yang dilakukan oleh warga di Kabupaten Pati, Jawa Tengah (terutama yang dimotori oleh Aliansi Masyarakat Pati Bersatu), dipicu oleh beberapa faktor utama berikut:

1. **Kenaikan Tarif PBB yang Drastis (Pemicu Awal)**
   Penyebab utama munculnya kemarahan warga adalah kebijakan Bupati Pati, Sudewo, yang menaikkan tarif Pajak Bumi dan Bangunan Perdesaan dan Perkotaan (PBB-P2) hingga mencapai **250 persen** (penyesuaian Nilai Jual Objek Pajak/NJOP). Kebijakan ini dinilai sangat memberatkan masyarakat, terutama di tengah kondisi ekonomi yang sedang sulit.

2. **Kebijakan Sempat Dibatalkan, tetapi Warga Tetap Kecewa**
   Meskipun gelombang protes membuat Bupati Pati akhirnya resmi membatalkan kenaikan PBB tersebut, massa tetap melanjutkan aksi unjuk rasa. Pembatalan itu tidak serta-merta meredakan kemarahan warga.

3. **Tuntutan Agar Bupati Mundur dan Dugaan Sikap Arogan**
   Seiring 

### user:

apa penyebab terjadinya demo di pati, jawa tengah?

...

### user:

Function response: { will_continue=None scheduling=None parts=None id=None name='exa_search_and_contents' response={'result': 'Title: Penyebab Warga Masih Kesal kepada Bupati Pati Sudewo  | tempo.co\nURL: https://www.tempo.co/politik/penyebab-warga-masih-kesal-kepada-bupati-pati-sudewo--2057082\nContent: Penyebab Warga Masih Kesal kepada Bupati Pati Sudewo | tempo.co\n\nUntuk Apa Indonesia Menerima Hibah Kapal Bekas Italia Alasan Kemhan Tak Beli Kapal Induk Baru Perang AS-Iran, KBRI Riyadh Minta WNI Lapor Diri Modernisasi Kapal Induk Garibaldi Dilakukan di Indonesia\n\nIklan\n\nNusa\n\n# Penyebab Warga Masih Kesal kepada Bupati Pati Sudewo\n\nPembatalan kenaikan PBB itu tak menyurutkan kekesalan warga Pati yang mengaku kecewa terhadap sikap Bupati Pati Sudewo yang dianggap arogan.\n\n10 Agustus 2025 | 10.45 WIB\n\nDengarkan artikel\n\nBagikan\n\nGabung Tempo Circle\n\nPerbesar\n\nWarga duduk di ruang tunggu saat antre membayar Pajak Bumi dan Bangunan (PBB) di Kantor Kecamatan Ngaliyan,

### model:

Demonstrasi besar-besaran yang dilakukan oleh warga di Kabupaten Pati, Jawa Tengah (terutama yang dimotori oleh Aliansi Masyarakat Pati Bersatu), dipicu oleh beberapa faktor utama berikut:

1. **Kenaikan Tarif PBB yang Drastis (Pemicu Awal)**
   Penyebab utama munculnya kemarahan warga adalah kebijakan Bupati Pati, Sudewo, yang menaikkan tarif Pajak Bumi dan Bangunan Perdesaan dan Perkotaan (PBB-P2) hingga mencapai **250 persen** (penyesuaian Nilai Jual Objek Pajak/NJOP). Kebijakan ini dinilai sangat memberatkan masyarakat, terutama di tengah kondisi ekonomi yang sedang sulit.

2. **Kebijakan Sempat Dibatalkan, tetapi Warga Tetap Kecewa**
   Meskipun gelombang protes membuat Bupati Pati akhirnya resmi membatalkan kenaikan PBB tersebut, massa tetap melanjutkan aksi unjuk rasa. Pembatalan itu tidak serta-merta meredakan kemarahan warga.

3. **Tuntutan Agar Bupati Mundur dan Dugaan Sikap Arogan**
   Seiring berjalannya waktu, tuntutan demonstrasi bergeser. Warga yang tergabung dalam aliansi masyarakat menilai Bupati Pati bersikap arogan dalam memimpin dan merespons keluhan warga. Oleh karena itu, unjuk rasa dilanjutkan dengan tuntutan agar Bupati Sudewo **mengundurkan diri/lengser** dari jabatannya, serta mendesak DPRD untuk memproses pemakzulan (melalui Pansus Hak Angket) dan mengusut dugaan kasus korupsi.

*(Sumber referensi: Kompas.id, Tempo, dan Kompas.com)*

--------------------------------------------------------------------------------
